# NumCompute-Stream — progressive decision-tree demonstration

This notebook loads a real CSV using the custom I/O reader, processes it in chunks, incrementally updates a NumPy preprocessing pipeline, and periodically retrains a single buffered tree. Every chunk is **evaluated before learning** to avoid optimistic train-on-test accuracy. The code uses only NumPy, matplotlib and the Python standard library.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from numcompute_stream import (
    DecisionTreeClassifier, SimpleImputer, StandardScaler,
    Pipeline, StreamTrainer, iter_csv_chunks,
)
from numcompute_stream.visualise import (
    plot_metric_over_time, compare_models, plot_predictions_vs_ground_truth,
)
# Works from either the repository root or inside demo/.
DATASET = Path("demo/dataset/test_student_performance_data.csv")
if not DATASET.exists():
    DATASET = Path("dataset/test_student_performance_data.csv")
FEATURES = ["Age", "StudyTimeWeekly", "Absences", "GPA"]
assert DATASET.exists(), "CSV dataset not found"

## 1. Build two streaming pipelines

Each learner has the same model but a different **buffer_size**. Imputation and scaling learn from each chunk; the pipeline stores raw chunks and applies a consistent transform at retraining. Until that point, predictions use the last fitted transformer snapshot.

In [ ]:
def make_trainer(buffer_size):
    pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="mean")),
        ("scale", StandardScaler()),
        ("tree", DecisionTreeClassifier(max_depth=5, min_samples_split=4,
                                        buffer_size=buffer_size, criterion="gini")),
    ])
    return StreamTrainer(pipeline, window_size=160)

small = make_trainer(80)
large = make_trainer(320)

## 2. Read CSV in chunks and progressively evaluate/train

A buffer retrains only when it collects enough observations. Early chunks cannot be scored because no model exists yet, so their accuracy is `NaN`, not zero.

In [ ]:
for X_chunk, y_chunk in iter_csv_chunks(DATASET, chunk_size=40,
                                        feature_columns=FEATURES, target_column="GradeClass",
                                        skip_missing_targets=True):
    small.process_chunk(X_chunk, y_chunk)
    large.process_chunk(X_chunk, y_chunk)

for name, trainer in (("Buffer 80", small), ("Buffer 320", large)):
    outcome = trainer.metrics.result()
    print(name, "| evaluated samples:", outcome["n_samples"],
          "| cumulative accuracy:", round(outcome["accuracy"], 4),
          "| retrain count:", trainer.n_retrains)

print("Latest chunk log:", small.logs[-1])
print("Cumulative confusion matrix:\n", small.metrics.result()["confusion_matrix"])

## 3. Visualise cumulative accuracy and buffer-size comparison

The data is held constant; only buffering configuration changes. Buffer sizes can change how many initial samples are scored, so comparisons should report evaluated sample counts.

In [ ]:
small_curve = [row["cumulative_accuracy"] for row in small.logs]
large_curve = [row["cumulative_accuracy"] for row in large.logs]
fig, ax = plot_metric_over_time(small_curve, "Cumulative accuracy — buffer 80", "Accuracy")
plt.show()
fig, ax = compare_models(small_curve, large_curve, ("Buffer 80", "Buffer 320"),
                         title="Buffered tree comparison", ylabel="Cumulative accuracy")
plt.show()

## 4. Latest unseen-chunk predictions vs ground truth

In [ ]:
if small.last_y_true_ is not None:
    fig, ax = plot_predictions_vs_ground_truth(small.last_y_true_, small.last_y_pred_)
    plt.show()

## 5. Verify stream semantics and runtime metrics

Log entries include retrain flags, per-chunk fitting seconds, buffered NumPy bytes and traced peak memory bytes. The memory measurements are not equivalent to whole-process RSS.

In [ ]:
assert small.n_retrains > large.n_retrains
assert 0 <= small.metrics.result()["accuracy"] <= 1
print("Small buffer retrains:", small.n_retrains)
print("Large buffer retrains:", large.n_retrains)
print("Peak observed buffer bytes:", max(row["buffer_memory_bytes"] for row in small.logs))
print("Example streaming prediction:", small.model.predict(np.array([[17., 12., 7., 3.1]])))